## Setup

In [1]:
!python3 -m pip install pyarango
!python3 -m pip install "python-arango>=5.0"


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import json
import requests
import sys
import time                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

from arango.client import ArangoClient

client = ArangoClient(hosts="http://localhost:8529")

db = client.db("db_test", username="root", password="test")       

aql = db.aql

## Import data

In [5]:
insert_query = """
LET data = [
    { "name": "Ned", "surname": "Stark", "alive": true, "age": 41, "traits": ["A","H","C","N","P"] },
    { "name": "Robert", "surname": "Baratheon", "alive": false, "traits": ["A","H","C"] },
    { "name": "Jaime", "surname": "Lannister", "alive": true, "age": 36, "traits": ["A","F","B"] },
    { "name": "Catelyn", "surname": "Stark", "alive": false, "age": 40, "traits": ["D","H","C"] },
    { "name": "Cersei", "surname": "Lannister", "alive": true, "age": 36, "traits": ["H","E","F"] },
    { "name": "Daenerys", "surname": "Targaryen", "alive": true, "age": 16, "traits": ["D","H","C"] },
    { "name": "Jorah", "surname": "Mormont", "alive": false, "traits": ["A","B","C","F"] },
    { "name": "Petyr", "surname": "Baelish", "alive": false, "traits": ["E","G","F"] },
    { "name": "Viserys", "surname": "Targaryen", "alive": false, "traits": ["O","L","N"] },
    { "name": "Jon", "surname": "Snow", "alive": true, "age": 16, "traits": ["A","B","C","F"] },
    { "name": "Sansa", "surname": "Stark", "alive": true, "age": 13, "traits": ["D","I","J"] },
    { "name": "Arya", "surname": "Stark", "alive": true, "age": 11, "traits": ["C","K","L"] },
    { "name": "Robb", "surname": "Stark", "alive": false, "traits": ["A","B","C","K"] },
    { "name": "Theon", "surname": "Greyjoy", "alive": true, "age": 16, "traits": ["E","R","K"] },
    { "name": "Bran", "surname": "Stark", "alive": true, "age": 10, "traits": ["L","J"] },
    { "name": "Joffrey", "surname": "Baratheon", "alive": false, "age": 19, "traits": ["I","L","O"] },
    { "name": "Sandor", "surname": "Clegane", "alive": true, "traits": ["A","P","K","F"] },
    { "name": "Tyrion", "surname": "Lannister", "alive": true, "age": 32, "traits": ["F","K","M","N"] },
    { "name": "Khal", "surname": "Drogo", "alive": false, "traits": ["A","C","O","P"] },
    { "name": "Tywin", "surname": "Lannister", "alive": false, "traits": ["O","M","H","F"] },
    { "name": "Davos", "surname": "Seaworth", "alive": true, "age": 49, "traits": ["C","K","P","F"] },
    { "name": "Samwell", "surname": "Tarly", "alive": true, "age": 17, "traits": ["C","L","I"] },
    { "name": "Stannis", "surname": "Baratheon", "alive": false, "traits": ["H","O","P","M"] },
    { "name": "Melisandre", "alive": true, "traits": ["G","E","H"] },
    { "name": "Margaery", "surname": "Tyrell", "alive": false, "traits": ["M","D","B"] },
    { "name": "Jeor", "surname": "Mormont", "alive": false, "traits": ["C","H","M","P"] },
    { "name": "Bronn", "alive": true, "traits": ["K","E","C"] },
    { "name": "Varys", "alive": true, "traits": ["M","F","N","E"] },
    { "name": "Shae", "alive": false, "traits": ["M","D","G"] },
    { "name": "Talisa", "surname": "Maegyr", "alive": false, "traits": ["D","C","B"] },
    { "name": "Gendry", "alive": false, "traits": ["K","C","A"] },
    { "name": "Ygritte", "alive": false, "traits": ["A","P","K"] },
    { "name": "Tormund", "surname": "Giantsbane", "alive": true, "traits": ["C","P","A","I"] },
    { "name": "Gilly", "alive": true, "traits": ["L","J"] },
    { "name": "Brienne", "surname": "Tarth", "alive": true, "age": 32, "traits": ["P","C","A","K"] },
    { "name": "Ramsay", "surname": "Bolton", "alive": true, "traits": ["E","O","G","A"] },
    { "name": "Ellaria", "surname": "Sand", "alive": true, "traits": ["P","O","A","E"] },
    { "name": "Daario", "surname": "Naharis", "alive": true, "traits": ["K","P","A"] },
    { "name": "Missandei", "alive": true, "traits": ["D","L","C","M"] },
    { "name": "Tommen", "surname": "Baratheon", "alive": true, "traits": ["I","L","B"] },
    { "name": "Jaqen", "surname": "H'ghar", "alive": true, "traits": ["H","F","K"] },
    { "name": "Roose", "surname": "Bolton", "alive": true, "traits": ["H","E","F","A"] },
    { "name": "The High Sparrow", "alive": true, "traits": ["H","M","F","O"] }
]

FOR d IN data
    INSERT d INTO Characters
"""

aql.execute(insert_query)

<Cursor>

## Join
The character data we imported has an attribute traits for each character, which is an array of strings. It does not store character features directly however:

In [6]:
find_ned_query = """
FOR c IN Characters
    FILTER c.name == "Ned"
    RETURN {"Name": c.name, "Traits": c.traits}
"""

query_result = aql.execute(find_ned_query)

for doc in  query_result:
    print(doc)
    print()

{'Name': 'Ned', 'Traits': ['A', 'H', 'C', 'N', 'P']}



Traits are rather a list of letters without an apparent meaning. The idea here is that traits is supposed to store documents keys of another collection, which we can use to resolve the letters to labels such as “strong”. The benefit of using another collection for the actual traits is, that we can easily query for all existing traits later on and store labels in multiple languages for instance in a central place. If we would embed traits directly…

```json
JSON({
    "Name": "Ned",
    "Traits": [
        {
            "de": "stark",
            "en": "strong"
        },
        {
            "de": "einflussreich",
            "en": "powerful"
        },
        {
            "de": "loyal",
            "en": "loyal"
        },
        {
            "de": "rational",
            "en": "rational"
        },
        {
            "de": "mutig",
            "en": "brave"
        }
    ]
})
```

… it becomes really hard to maintain traits. If you were to rename or translate one of them, you would need to find all other character documents with the same trait and perform the changes there too. If we only refer to a trait in another collection, it is as easy as updating a single document.

![locations](https://github.com/arangodb/interactive_tutorials/blob/master/notebooks/img/join.png?raw=1)

## Traits Collection

In [8]:
if not db.has_collection(name="Traits"):
    db.create_collection(name="Traits")


In [9]:
insert_query = """
LET data = [
    { "_key": "A", "en": "strong", "de": "stark" },
    { "_key": "B", "en": "polite", "de": "freundlich" },
    { "_key": "C", "en": "loyal", "de": "loyal" },
    { "_key": "D", "en": "beautiful", "de": "schön" },
    { "_key": "E", "en": "sneaky", "de": "hinterlistig" },
    { "_key": "F", "en": "experienced", "de": "erfahren" },
    { "_key": "G", "en": "corrupt", "de": "korrupt" },
    { "_key": "H", "en": "powerful", "de": "einflussreich" },
    { "_key": "I", "en": "naive", "de": "naiv" },
    { "_key": "J", "en": "unmarried", "de": "unverheiratet" },
    { "_key": "K", "en": "skillful", "de": "geschickt" },
    { "_key": "L", "en": "young", "de": "jung" },
    { "_key": "M", "en": "smart", "de": "klug" },
    { "_key": "N", "en": "rational", "de": "rational" },
    { "_key": "O", "en": "ruthless", "de": "skrupellos" },
    { "_key": "P", "en": "brave", "de": "mutig" },
    { "_key": "Q", "en": "mighty", "de": "mächtig" },
    { "_key": "R", "en": "weak", "de": "schwach" }
]

FOR d IN data
    INSERT d INTO Traits
"""

aql.execute(insert_query)

<Cursor>

In [12]:
all_traits = """
FOR t IN Traits
    RETURN t
"""

query_result = aql.execute(all_traits, count=True)

print(f"{len(query_result)} Traits")
for doc in  query_result:
    print(doc)
    print()

18 Traits
{'_key': 'A', '_id': 'Traits/A', '_rev': '_ldvorMC---', 'en': 'strong', 'de': 'stark'}

{'_key': 'B', '_id': 'Traits/B', '_rev': '_ldvorMC--_', 'en': 'polite', 'de': 'freundlich'}

{'_key': 'C', '_id': 'Traits/C', '_rev': '_ldvorMC--A', 'en': 'loyal', 'de': 'loyal'}

{'_key': 'D', '_id': 'Traits/D', '_rev': '_ldvorMC--B', 'en': 'beautiful', 'de': 'schön'}

{'_key': 'E', '_id': 'Traits/E', '_rev': '_ldvorMC--C', 'en': 'sneaky', 'de': 'hinterlistig'}

{'_key': 'F', '_id': 'Traits/F', '_rev': '_ldvorMC--D', 'en': 'experienced', 'de': 'erfahren'}

{'_key': 'G', '_id': 'Traits/G', '_rev': '_ldvorMC--E', 'en': 'corrupt', 'de': 'korrupt'}

{'_key': 'H', '_id': 'Traits/H', '_rev': '_ldvorMC--F', 'en': 'powerful', 'de': 'einflussreich'}

{'_key': 'I', '_id': 'Traits/I', '_rev': '_ldvorMC--G', 'en': 'naive', 'de': 'naiv'}

{'_key': 'J', '_id': 'Traits/J', '_rev': '_ldvorMC--H', 'en': 'unmarried', 'de': 'unverheiratet'}

{'_key': 'K', '_id': 'Traits/K', '_rev': '_ldvorMC--I', 'en': 'ski

## Joining Traits

Let’s start simple by returning only the traits attribute of each character:

In [ ]:
all_characters_traits = """
FOR c IN Characters
    RETURN c.traits
"""

query_result = aql.execute(all_characters_traits)

for doc in  query_result:
    print(doc)
    print()

['A', 'H', 'C', 'N', 'P']

['A', 'H', 'C']

['A', 'F', 'B']

['D', 'H', 'C']

['H', 'E', 'F']

['D', 'H', 'C']

['A', 'B', 'C', 'F']

['E', 'G', 'F']

['O', 'L', 'N']

['A', 'B', 'C', 'F']

['D', 'I', 'J']

['C', 'K', 'L']

['A', 'B', 'C', 'K']

['E', 'R', 'K']

['L', 'J']

['I', 'L', 'O']

['A', 'P', 'K', 'F']

['F', 'K', 'M', 'N']

['A', 'C', 'O', 'P']

['O', 'M', 'H', 'F']

['C', 'K', 'P', 'F']

['C', 'L', 'I']

['H', 'O', 'P', 'M']

['G', 'E', 'H']

['M', 'D', 'B']

['C', 'H', 'M', 'P']

['K', 'E', 'C']

['M', 'F', 'N', 'E']

['M', 'D', 'G']

['D', 'C', 'B']

['K', 'C', 'A']

['A', 'P', 'K']

['C', 'P', 'A', 'I']

['L', 'J']

['P', 'C', 'A', 'K']

['E', 'O', 'G', 'A']

['P', 'O', 'A', 'E']

['K', 'P', 'A']

['D', 'L', 'C', 'M']

['I', 'L', 'B']

['H', 'F', 'K']

['H', 'E', 'F', 'A']

['H', 'M', 'F', 'O']



In [14]:
all_characters_traits = """
FOR c IN Characters
    LIMIT 5
    RETURN DOCUMENT("Traits", c.traits)
"""

query_result = aql.execute(all_characters_traits)

for doc in  query_result:
    print(doc)
    print()

[{'_key': 'A', '_id': 'Traits/A', '_rev': '_ldvorMC---', 'en': 'strong', 'de': 'stark'}, {'_key': 'H', '_id': 'Traits/H', '_rev': '_ldvorMC--F', 'en': 'powerful', 'de': 'einflussreich'}, {'_key': 'C', '_id': 'Traits/C', '_rev': '_ldvorMC--A', 'en': 'loyal', 'de': 'loyal'}, {'_key': 'N', '_id': 'Traits/N', '_rev': '_ldvorMC--L', 'en': 'rational', 'de': 'rational'}, {'_key': 'P', '_id': 'Traits/P', '_rev': '_ldvorMC--N', 'en': 'brave', 'de': 'mutig'}]

[{'_key': 'A', '_id': 'Traits/A', '_rev': '_ldvorMC---', 'en': 'strong', 'de': 'stark'}, {'_key': 'H', '_id': 'Traits/H', '_rev': '_ldvorMC--F', 'en': 'powerful', 'de': 'einflussreich'}, {'_key': 'C', '_id': 'Traits/C', '_rev': '_ldvorMC--A', 'en': 'loyal', 'de': 'loyal'}]

[{'_key': 'A', '_id': 'Traits/A', '_rev': '_ldvorMC---', 'en': 'strong', 'de': 'stark'}, {'_key': 'F', '_id': 'Traits/F', '_rev': '_ldvorMC--D', 'en': 'experienced', 'de': 'erfahren'}, {'_key': 'B', '_id': 'Traits/B', '_rev': '_ldvorMC--_', 'en': 'polite', 'de': 'freund

## Joining Characters and Traits
The `DOCUMENT()` function utilizes primary indices to look up documents quickly. It is limited to find documents via their identifiers however. For a use case like in our example it is sufficient to accomplish a simple join.

There is another, more flexible syntax for joins: nested `FOR` loops over multiple collections, with a `FILTER` condition to match up attributes. In case of the traits key array, there needs to be a third loop to iterate over the keys:

In [15]:
all_characters_traits = """
FOR c IN Characters
  LIMIT 5
  RETURN MERGE(c, {
    traits: (
      FOR key IN c.traits
        FOR t IN Traits
          FILTER t._key == key
          RETURN t.en
    )
  })
"""

query_result = aql.execute(all_characters_traits)

for doc in  query_result:
    print(doc)
    print()  

{'_id': 'Characters/23115', '_key': '23115', '_rev': '_ldvjJyW---', 'age': 41, 'alive': True, 'name': 'Ned', 'surname': 'Stark', 'traits': ['strong', 'powerful', 'loyal', 'rational', 'brave']}

{'_id': 'Characters/23116', '_key': '23116', '_rev': '_ldvjJyW--_', 'alive': False, 'name': 'Robert', 'surname': 'Baratheon', 'traits': ['strong', 'powerful', 'loyal']}

{'_id': 'Characters/23117', '_key': '23117', '_rev': '_ldvjJyW--A', 'age': 36, 'alive': True, 'name': 'Jaime', 'surname': 'Lannister', 'traits': ['strong', 'experienced', 'polite']}

{'_id': 'Characters/23118', '_key': '23118', '_rev': '_ldvjJyW--B', 'age': 40, 'alive': False, 'name': 'Catelyn', 'surname': 'Stark', 'traits': ['beautiful', 'powerful', 'loyal']}

{'_id': 'Characters/23119', '_key': '23119', '_rev': '_ldvjJyW--C', 'age': 36, 'alive': True, 'name': 'Cersei', 'surname': 'Lannister', 'traits': ['powerful', 'sneaky', 'experienced']}



For more about `JOIN and MERGE`, [here](https://docs.arango.ai/arangodb/stable/get-started/start-using-aql/joins/)